# Vector Databases

We will look at the following Vector Databases in this course:
- [AstraDB](#astradb-from-datastax)
- [Pinecone](#pinecone)

### AstraDB (from datastax)
Go from app idea to production with the AI Platform with Astra DB, the ultra-low latency database made for AI and Langflow, the low-code RAG IDE
https://www.datastax.com/

In [ ]:
# create a Vector database in AstraDB named rag_db_Mrinmoy
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from langchain_openai import OpenAIEmbeddings
from langchain_astradb import AstraDBVectorStore
from langchain_core.documents import Document

In [2]:
openai_key = os.getenv("OPENAI_API_KEY")
if openai_key is not None:
	os.environ["OPENAI_API_KEY"] = openai_key
else:
	raise ValueError("OPENAI_API_KEY environment variable is not set.")

In [ ]:
embeddings=OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)

In [6]:
vectorstore = AstraDBVectorStore(
    collection_name="rag_db_Mrinmoy",
    embedding=embeddings,
    api_endpoint=os.getenv("ASTRA_DB_API_ENDPOINT"),
    token = os.getenv("ASTRA_DB_APPLICATION_TOKEN"),
    namespace=None
)

In [8]:
document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [9]:
vectorstore.add_documents(documents = documents)

['e8f5ad191ded48878faeb967d1bc4b96',
 '3acf280172fc466b86b3f506eb65f447',
 'f0c8c15071fb4ef2abd4824c74197ec6',
 'ff7f45f80b064996a51c943c023d5499',
 '48c59555598645bcaa86bc7d674f3eb4',
 '5619b647fd1341a881b17128a91f9cd3',
 '1939b79f8df946759f7b3bd7afc2356c',
 '6f6db5d42e8f496c8e12333447337a4c',
 'a94dd3720ea14996a5393268c1065421',
 '480243038d01461e93540085ae2ad2ed']

In [10]:
# search from vector database
query  = "What is the waether?"
vectorstore.similarity_search(query, k=4)

[Document(id='3acf280172fc466b86b3f506eb65f447', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='480243038d01461e93540085ae2ad2ed', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :('),
 Document(id='a94dd3720ea14996a5393268c1065421', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.'),
 Document(id='48c59555598645bcaa86bc7d674f3eb4', metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again.")]

In [12]:
results = vectorstore.similarity_search(
   "Langchain provides abstractions to make working with LLMs easier.", 
   k=3,
   filter = {"source": "tweet"}
)

for res in results:
    print(f'* "{res.page_content}", metadata = {res.metadata}')

* "LangGraph is the best framework for building stateful, agentic applications!", metadata = {'source': 'tweet'}
* "Building an exciting new project with LangChain - come check it out!", metadata = {'source': 'tweet'}
* "I have a bad feeling I am going to get deleted :(", metadata = {'source': 'tweet'}


In [13]:
results = vectorstore.similarity_search_with_score(
   "Langchain provides abstractions to make working with LLMs easier.", 
   k=3,
   filter = {"source": "tweet"}
)

for res, score in results:
    print(f'* [SIM={score:.2f}] "{res.page_content}", metadata = {res.metadata}')

* [SIM=0.71] "LangGraph is the best framework for building stateful, agentic applications!", metadata = {'source': 'tweet'}
* [SIM=0.69] "Building an exciting new project with LangChain - come check it out!", metadata = {'source': 'tweet'}
* [SIM=0.52] "I have a bad feeling I am going to get deleted :(", metadata = {'source': 'tweet'}


In [14]:
# convert vectorstore to retriver
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold", 
    search_kwargs={"k": 1, "score_threshold": 0.5}
)

retriever.invoke("Stealing from bank is a crime", filter = {"source": "news"})

[Document(id='ff7f45f80b064996a51c943c023d5499', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

In [15]:
# with mmr search
retriever = vectorstore.as_retriever(
    search_type="mmr", 
    search_kwargs={"k": 1}
)

retriever.invoke("Stealing from bank is a crime", filter = {"source": "news"})

[Document(id='ff7f45f80b064996a51c943c023d5499', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

## Pinecone 
Create an account here https://www.pinecone.io/ and get the api key

In [17]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [27]:
from pinecone import Pinecone
from pinecone import ServerlessSpec
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document

In [20]:
openai_key = os.getenv("OPENAI_API_KEY")
if openai_key is not None:
	os.environ["OPENAI_API_KEY"] = openai_key
else:
	raise ValueError("OPENAI_API_KEY environment variable is not set.")

In [18]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

In [21]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)

In [24]:
index_name = "rag-mrinmoy"

if not pc.has_index(index_name):
    index = pc.create_index(
        name = index_name,
        dimension = 1024,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [26]:
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings
)

In [28]:
document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [29]:
vectorstore.add_documents(documents = documents)

['05fb83e1-7eaa-44a3-b2a9-3888b919e45b',
 '461e0b50-271a-4d2c-85ef-ebc98102c21b',
 '171f3da7-db1c-44dc-a254-05b938636103',
 '7bfce9a8-15d0-4335-985f-bf636622200e',
 'd560ccd0-ae76-46b3-b6c0-4728156d772d',
 '48e40ffa-095c-436e-9acd-5c86f47c8e21',
 '31207705-a068-44ec-b26c-50817ed1a7c8',
 '389240ce-dd6b-4001-be4d-00468b8fda24',
 'f022ec54-2bc8-45b4-a616-ec7bd1672105',
 'f6119f6c-07e6-4527-a260-0fecc018a480']

In [30]:
results = vectorstore.similarity_search(
   "Langchain provides abstractions to make working with LLMs easier.", 
   k=3,
   filter = {"source": "tweet"}
)

for res in results:
    print(f'* "{res.page_content}", metadata = {res.metadata}')

* "LangGraph is the best framework for building stateful, agentic applications!", metadata = {'source': 'tweet'}
* "Building an exciting new project with LangChain - come check it out!", metadata = {'source': 'tweet'}
* "I have a bad feeling I am going to get deleted :(", metadata = {'source': 'tweet'}


In [31]:
results = vectorstore.similarity_search_with_score(
   "Langchain provides abstractions to make working with LLMs easier.", 
   k=3,
   filter = {"source": "tweet"}
)

for res, score in results:
    print(f'* [SIM={score:.2f}] "{res.page_content}", metadata = {res.metadata}')

* [SIM=0.42] "LangGraph is the best framework for building stateful, agentic applications!", metadata = {'source': 'tweet'}
* [SIM=0.38] "Building an exciting new project with LangChain - come check it out!", metadata = {'source': 'tweet'}
* [SIM=0.05] "I have a bad feeling I am going to get deleted :(", metadata = {'source': 'tweet'}


In [32]:
# convert vectorstore to retriver
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold", 
    search_kwargs={"k": 1, "score_threshold": 0.5}
)

retriever.invoke("Stealing from bank is a crime", filter = {"source": "news"})

[Document(id='7bfce9a8-15d0-4335-985f-bf636622200e', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]